In [ ]:
from huggingface_hub import login
from dotenv import load_dotenv
import os

In [ ]:
load_dotenv(override=True)

hf_token = os.getenv('HF_TOKEN')
login(token=hf_token)

In [ ]:
from datasets import load_dataset

ds = load_dataset("ayanmukherjee/repliqa-4-company-policies-answerable")

In [ ]:
ds

In [ ]:
repliqa_4_ds = ds['repliqa_4']

In [ ]:
repliqa_4_ds

In [ ]:
total_count = 0
unans_count = 0

In [ ]:
for dataset in ds:
    for item in ds[dataset]:
        if item['long_answer'] == 'NA' or item['answer'] == 'The answer is not found in the document.':
            unans_count += 1
        total_count += 1

In [ ]:
print(unans_count)
print(total_count)

In [ ]:
ds[1000]

In [ ]:
topic_counts = {}

for item in repliqa_4_ds:
    if topic_counts.get(item['document_topic'], "") == "":
        topic_counts[item['document_topic']] = 0
    else:
        topic_counts[item['document_topic']] += 1

In [ ]:
topic_counts

In [ ]:

from langchain_core.documents import Document

documents = []

for split in ds: 
    for item in ds[split]:

        documents.append(
            Document(
                page_content=item.get("document_extracted", "").strip(),
                metadata={
                    "document_id": item.get("document_id"),
                    "topic": item.get("document_topic"),
                    "source": item.get("document_path")
                }
            )
        )


In [ ]:
len(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

In [ ]:
len(chunks)

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
DB_NAME = "chroma_langchain_db"

In [ ]:
print(os.path.exists(DB_NAME))

In [ ]:
if os.path.exists(DB_NAME):
    Chroma(embedding_function=embeddings, persist_directory=DB_NAME).delete_collection()

In [ ]:
vector_store = Chroma(
    collection_name="context_collection",
    embedding_function=embeddings,
    persist_directory=DB_NAME
)
print(f"Vectorstore created with {vector_store._collection.count()} documents")

In [ ]:
vector_store.add_documents(documents=chunks)

Using all the splits was taking too much time to load in the vector db, >31. So moving to single split. Batch size error if random number of documents is selected. So, based on topics, we selected "Company Policies" because low number of documents.

In [ ]:
# Let's investigate the vectors

collection = vector_store._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")